# 21 malignant state pseudobulk

**Purpose:** Run malignant-state pseudobulk differential expression.

**Original analysis notebook:** `20_malignant_state_pseudobulk_DE_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 39 — Malignant-state pseudobulk differential expression / marker refinement

**Goal:** refine and validate malignant-state marker programs using sample-level pseudobulk expression.

This notebook uses the final annotated object with malignant-state labels and CopyKAT consensus. It aggregates raw counts by **sample × malignant_state**, normalizes pseudobulk profiles to logCPM, and performs **state-vs-rest pseudobulk DE** among malignant-like states.

Primary use:
- identify robust marker genes for each malignant state;
- confirm that malignant-state labels are supported by pseudobulk expression, not only single-cell cluster markers;
- produce state-level marker tables and heatmaps for thesis figures.

Important: this is **pseudobulk state marker refinement**, not full edgeR/DESeq2 modeling. The unit of inference is sample-level pseudobulk, which is more defensible than treating every cell as independent.


In [ ]:
# =========================
# Imports and configuration
# =========================

from pathlib import Path
import os
import gc
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import mannwhitneyu

import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures" / "malignant_state_pseudobulk_DE_HVG8000_r0_6"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
FALLBACK_INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation.h5ad"

PSEUDOBULK_H5AD = PROCESSED_DIR / "GBM_core_4datasets_malignant_state_pseudobulk_logCPM.h5ad"

# Main output paths
READINESS_TSV = METADATA_DIR / "malignant_pseudobulk_DE_readiness_HVG8000_r0_6.tsv"
PSEUDOBULK_QC_TSV = METADATA_DIR / "malignant_pseudobulk_sample_state_QC_HVG8000_r0_6.tsv"
STATE_SUMMARY_TSV = METADATA_DIR / "malignant_pseudobulk_state_summary_HVG8000_r0_6.tsv"
DE_TOP100_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_state_vs_rest_top100_up_down_HVG8000_r0_6.tsv"
DE_SIGNIFICANT_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_state_vs_rest_significant_HVG8000_r0_6.tsv"
DE_ALL_SUMMARY_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_state_vs_rest_test_summary_HVG8000_r0_6.tsv"
REFINEMENT_SUMMARY_TSV = METADATA_DIR / "malignant_state_marker_refinement_summary_HVG8000_r0_6.tsv"
FIGURE_MANIFEST_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_figure_manifest_HVG8000_r0_6.tsv"
OUTPUT_MANIFEST_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_output_manifest_HVG8000_r0_6.tsv"

# Conservative defaults
MIN_CELLS_PER_PSEUDOBULK = 30
MIN_PSEUDOBULKS_PER_GROUP = 3
MIN_DETECTED_FRACTION_IN_STATE = 0.10
MIN_ABS_LOGFC_FOR_REPORT = 0.25
SIGNIFICANT_PADJ = 0.05
TOP_N_PER_DIRECTION = 100
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("Input h5ad:", INPUT_H5AD)
print("Output pseudobulk h5ad:", PSEUDOBULK_H5AD)


In [ ]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find {label}. Tried: {candidates}. Available columns include: {list(df.columns)[:80]}")
    return None


def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    mask = np.isfinite(pvals)
    if mask.sum() == 0:
        return out
    p = pvals[mask]
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    tmp = np.empty(n)
    tmp[order] = q
    out[mask] = tmp
    return out


def fdr_adjust(pvals):
    if HAS_STATSMODELS:
        p = np.asarray(pvals, dtype=float)
        out = np.full_like(p, np.nan, dtype=float)
        mask = np.isfinite(p)
        if mask.sum() > 0:
            out[mask] = multipletests(p[mask], method="fdr_bh")[1]
        return out
    return benjamini_hochberg(pvals)


def safe_mwu(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < MIN_PSEUDOBULKS_PER_GROUP or len(y) < MIN_PSEUDOBULKS_PER_GROUP:
        return np.nan
    if np.allclose(x, x[0]) and np.allclose(y, y[0]) and np.isclose(x[0], y[0]):
        return 1.0
    try:
        return float(mannwhitneyu(x, y, alternative="two-sided").pvalue)
    except Exception:
        return np.nan


def sanitize_filename(x):
    return str(x).replace("/", "_").replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")


In [ ]:
# =========================
# Load final malignant-state object
# =========================

if not INPUT_H5AD.exists():
    print(f"[WARNING] Main input not found: {INPUT_H5AD}")
    print(f"[TRYING FALLBACK] {FALLBACK_INPUT_H5AD}")
    INPUT_H5AD = FALLBACK_INPUT_H5AD

if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Input h5ad not found: {INPUT_H5AD}")

adata = sc.read_h5ad(INPUT_H5AD)
print(adata)
print("obs columns:", len(adata.obs.columns))
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys())[:20])


In [ ]:
# =========================
# Resolve metadata columns and malignant-state labels
# =========================

obs = adata.obs

sample_col = choose_first_existing_column(
    obs,
    ["sample_id", "sample", "sample_name", "orig.ident", "library_id", "Sample", "sampleID"],
    label="sample column"
)
condition_col = choose_first_existing_column(
    obs,
    ["condition", "Condition", "disease_condition", "diagnosis", "group"],
    label="condition column"
)
patient_col = choose_first_existing_column(
    obs,
    ["patient_id", "patient", "patientID", "donor_id", "donor", "case_id"],
    required=False,
    label="patient column"
)
cluster_col = choose_first_existing_column(
    obs,
    ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "cluster", "leiden"],
    label="cluster column"
)
compartment_col = choose_first_existing_column(
    obs,
    ["final_compartment_major", "final_malignancy_group_copykat_marker", "malignancy_group", "final_group"],
    required=False,
    label="final compartment column"
)
state_col = choose_first_existing_column(
    obs,
    ["malignant_state", "final_malignant_state", "malignant_state_final", "malignant_state_refined"],
    required=False,
    label="malignant state column"
)

print("sample_col:", sample_col)
print("condition_col:", condition_col)
print("patient_col:", patient_col)
print("cluster_col:", cluster_col)
print("compartment_col:", compartment_col)
print("state_col:", state_col)

# If malignant state column does not exist, derive it from cluster IDs.
if state_col is None:
    print("[INFO] malignant state column not found. Deriving malignant_state from cluster mapping.")
    cluster_as_str = obs[cluster_col].astype(str)
    mapping = {
        "0": "OPC_NPC_like_malignant",
        "4": "hypoxia_stress_malignant",
        "7": "AC_like_reactive_malignant",
        "11": "NPC_neuronal_like_malignant",
        "2": "cycling_malignant_candidate",
    }
    adata.obs["malignant_state"] = cluster_as_str.map(mapping).fillna("not_malignant_state")
    state_col = "malignant_state"

# Normalize state labels for consistent reporting while preserving original column.
state_norm_col = "malignant_state_for_pseudobulk"
state_raw = adata.obs[state_col].astype(str)
cluster_str = adata.obs[cluster_col].astype(str)

# Use cluster mapping as a safety override for the known malignant states.
cluster_state_mapping = {
    "0": "OPC_NPC_like_malignant",
    "4": "hypoxia_stress_malignant",
    "7": "AC_like_reactive_malignant",
    "11": "NPC_neuronal_like_malignant",
    "2": "cycling_malignant_candidate",
}
state_norm = state_raw.copy()
for cl, st in cluster_state_mapping.items():
    state_norm.loc[cluster_str == cl] = st

# Exclude everything not in the planned malignant-state universe.
valid_states = [
    "OPC_NPC_like_malignant",
    "hypoxia_stress_malignant",
    "AC_like_reactive_malignant",
    "NPC_neuronal_like_malignant",
    "cycling_malignant_candidate",
]
state_norm = state_norm.where(state_norm.isin(valid_states), "not_malignant_state")
adata.obs[state_norm_col] = pd.Categorical(state_norm, categories=valid_states + ["not_malignant_state"])

print(adata.obs[state_norm_col].value_counts(dropna=False))


In [ ]:
# =========================
# Readiness report
# =========================

if "counts" in adata.layers:
    counts_source = "layers['counts']"
    X_counts = adata.layers["counts"]
else:
    counts_source = "X"
    X_counts = adata.X

# Lightweight count sanity check
x_sample = X_counts[:1000, :1000]
if sp.issparse(x_sample):
    vals = x_sample.data
else:
    vals = np.asarray(x_sample).ravel()
vals = vals[np.isfinite(vals)]
integer_like = bool(np.allclose(vals, np.round(vals))) if vals.size else None
has_negative = bool(np.any(vals < 0)) if vals.size else None

readiness = pd.DataFrame([{
    "input_h5ad": str(INPUT_H5AD),
    "n_cells": adata.n_obs,
    "n_genes": adata.n_vars,
    "counts_source": counts_source,
    "counts_integer_like_sample": integer_like,
    "counts_has_negative_sample": has_negative,
    "sample_col": sample_col,
    "condition_col": condition_col,
    "patient_col": patient_col if patient_col is not None else "NA",
    "cluster_col": cluster_col,
    "compartment_col": compartment_col if compartment_col is not None else "NA",
    "state_col_original": state_col,
    "state_col_used": state_norm_col,
    "min_cells_per_pseudobulk": MIN_CELLS_PER_PSEUDOBULK,
    "min_pseudobulks_per_group": MIN_PSEUDOBULKS_PER_GROUP,
}])
write_tsv_replace(readiness, READINESS_TSV)
display(readiness)


In [ ]:
# =========================
# Select malignant-state cells and build pseudobulk groups
# =========================

mal_mask = adata.obs[state_norm_col].astype(str).isin(valid_states).values
adata_mal = adata[mal_mask].copy()
print(adata_mal)
print(adata_mal.obs[state_norm_col].value_counts())

# Build stable pseudobulk group IDs: sample × state.
pb_obs_source = adata_mal.obs[[sample_col, condition_col, state_norm_col]].copy()
if patient_col is not None:
    pb_obs_source[patient_col] = adata_mal.obs[patient_col].astype(str).values
else:
    pb_obs_source["patient_id_pseudobulk"] = pb_obs_source[sample_col].astype(str).values
    patient_col = "patient_id_pseudobulk"

pb_obs_source[sample_col] = pb_obs_source[sample_col].astype(str)
pb_obs_source[condition_col] = pb_obs_source[condition_col].astype(str)
pb_obs_source[patient_col] = pb_obs_source[patient_col].astype(str)
pb_obs_source[state_norm_col] = pb_obs_source[state_norm_col].astype(str)
pb_obs_source["pseudobulk_id"] = (
    pb_obs_source[sample_col] + "__" + pb_obs_source[state_norm_col]
)

# Group sizes.
group_counts = (
    pb_obs_source
    .groupby(["pseudobulk_id", sample_col, patient_col, condition_col, state_norm_col], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values([condition_col, sample_col, state_norm_col])
)

group_counts["passes_min_cells"] = group_counts["n_cells"] >= MIN_CELLS_PER_PSEUDOBULK
write_tsv_replace(group_counts, PSEUDOBULK_QC_TSV)
display(group_counts.head())
print("n pseudobulks total:", group_counts.shape[0])
print("n pseudobulks passing min cells:", int(group_counts["passes_min_cells"].sum()))


In [ ]:
# =========================
# Aggregate raw counts by sample × malignant state
# =========================

X_mal = adata_mal.layers["counts"] if "counts" in adata_mal.layers else adata_mal.X
if not sp.issparse(X_mal):
    X_mal = sp.csr_matrix(X_mal)
else:
    X_mal = X_mal.tocsr()

pb_ids = pb_obs_source["pseudobulk_id"].values
unique_pb = group_counts["pseudobulk_id"].values
pb_index = {g: i for i, g in enumerate(unique_pb)}
cell_group_codes = np.array([pb_index[x] for x in pb_ids], dtype=np.int32)

agg_counts = []
for i, gid in enumerate(unique_pb):
    idx = np.where(cell_group_codes == i)[0]
    summed = np.asarray(X_mal[idx, :].sum(axis=0)).ravel()
    agg_counts.append(summed)
    if (i + 1) % 50 == 0 or (i + 1) == len(unique_pb):
        print(f"Aggregated {i+1}/{len(unique_pb)} pseudobulks")

pb_counts = np.vstack(agg_counts).astype(np.float32)
lib_size = pb_counts.sum(axis=1)
lib_size_safe = np.where(lib_size > 0, lib_size, 1.0)
log_cpm = np.log1p((pb_counts / lib_size_safe[:, None]) * 1e6).astype(np.float32)

pb_obs = group_counts.set_index("pseudobulk_id").copy()
pb_obs["library_size"] = lib_size
pb_obs["n_detected_genes"] = (pb_counts > 0).sum(axis=1)

pb_adata = ad.AnnData(
    X=log_cpm,
    obs=pb_obs,
    var=adata_mal.var.copy(),
)
pb_adata.layers["counts"] = pb_counts
pb_adata.uns["description"] = "Sample × malignant-state pseudobulk logCPM object. X=log1p(CPM), layers['counts']=raw pseudobulk counts."

write_h5ad_replace(pb_adata, PSEUDOBULK_H5AD)
print(pb_adata)

del adata_mal, X_mal, pb_counts, log_cpm, agg_counts
_gc = gc.collect()
print("gc:", _gc)


In [ ]:
# =========================
# State-level pseudobulk summary
# =========================

pb = pb_adata.copy()
pb.obs["passes_de_filter"] = pb.obs["n_cells"] >= MIN_CELLS_PER_PSEUDOBULK

state_summary = (
    pb.obs
    .groupby(state_norm_col, observed=True)
    .agg(
        n_pseudobulks=("passes_de_filter", "size"),
        n_pseudobulks_passing=("passes_de_filter", "sum"),
        total_cells=("n_cells", "sum"),
        median_cells_per_pseudobulk=("n_cells", "median"),
        n_samples=(sample_col, "nunique"),
        n_patients=(patient_col, "nunique"),
        n_conditions=(condition_col, "nunique"),
        median_library_size=("library_size", "median"),
        median_detected_genes=("n_detected_genes", "median"),
    )
    .reset_index()
    .rename(columns={state_norm_col: "malignant_state"})
)
write_tsv_replace(state_summary, STATE_SUMMARY_TSV)
display(state_summary)


In [ ]:
# =========================
# Pseudobulk DE: malignant state vs rest
# =========================

pb = pb_adata[pb_adata.obs["n_cells"].values >= MIN_CELLS_PER_PSEUDOBULK].copy()
expr = np.asarray(pb.X, dtype=np.float32)  # logCPM
counts = np.asarray(pb.layers["counts"], dtype=np.float32)
genes = pb.var_names.astype(str).values
states = pb.obs[state_norm_col].astype(str).values

all_results = []
test_summaries = []

for st in valid_states:
    mask_state = states == st
    mask_rest = ~mask_state
    n_state = int(mask_state.sum())
    n_rest = int(mask_rest.sum())
    cells_state = int(pb.obs.loc[mask_state, "n_cells"].sum())
    cells_rest = int(pb.obs.loc[mask_rest, "n_cells"].sum())
    print(f"[DE] {st}: n_pseudobulk_state={n_state}, n_rest={n_rest}")

    if n_state < MIN_PSEUDOBULKS_PER_GROUP or n_rest < MIN_PSEUDOBULKS_PER_GROUP:
        test_summaries.append({
            "malignant_state": st,
            "status": "skipped_insufficient_pseudobulks",
            "n_pseudobulk_state": n_state,
            "n_pseudobulk_rest": n_rest,
            "cells_state": cells_state,
            "cells_rest": cells_rest,
        })
        continue

    x = expr[mask_state, :]
    y = expr[mask_rest, :]
    cx = counts[mask_state, :]
    cy = counts[mask_rest, :]

    mean_state = x.mean(axis=0)
    mean_rest = y.mean(axis=0)
    median_state = np.median(x, axis=0)
    median_rest = np.median(y, axis=0)
    logfc = mean_state - mean_rest
    pct_state = (cx > 0).mean(axis=0)
    pct_rest = (cy > 0).mean(axis=0)
    pct_diff = pct_state - pct_rest

    pvals = np.empty(len(genes), dtype=float)
    for j in range(len(genes)):
        pvals[j] = safe_mwu(x[:, j], y[:, j])
        if (j + 1) % 5000 == 0:
            print(f"  tested {j+1}/{len(genes)} genes for {st}")

    padj = fdr_adjust(pvals)

    df = pd.DataFrame({
        "malignant_state": st,
        "gene": genes,
        "mean_logCPM_state": mean_state,
        "mean_logCPM_rest": mean_rest,
        "median_logCPM_state": median_state,
        "median_logCPM_rest": median_rest,
        "logFC_logCPM_state_vs_rest": logfc,
        "pct_pseudobulk_detected_state": pct_state,
        "pct_pseudobulk_detected_rest": pct_rest,
        "pct_detected_diff_state_minus_rest": pct_diff,
        "pval_mannwhitney": pvals,
        "padj_fdr_bh": padj,
        "n_pseudobulk_state": n_state,
        "n_pseudobulk_rest": n_rest,
        "cells_state": cells_state,
        "cells_rest": cells_rest,
    })
    df["direction"] = np.where(df["logFC_logCPM_state_vs_rest"] >= 0, "up_in_state", "down_in_state")
    df["passes_basic_marker_filter"] = (
        (df["pct_pseudobulk_detected_state"] >= MIN_DETECTED_FRACTION_IN_STATE) &
        (df["padj_fdr_bh"] <= SIGNIFICANT_PADJ) &
        (df["logFC_logCPM_state_vs_rest"].abs() >= MIN_ABS_LOGFC_FOR_REPORT)
    )

    all_results.append(df)
    test_summaries.append({
        "malignant_state": st,
        "status": "tested",
        "n_pseudobulk_state": n_state,
        "n_pseudobulk_rest": n_rest,
        "cells_state": cells_state,
        "cells_rest": cells_rest,
        "n_genes_tested": int(df.shape[0]),
        "n_significant_basic_filter": int(df["passes_basic_marker_filter"].sum()),
        "n_up_basic_filter": int(((df["passes_basic_marker_filter"]) & (df["logFC_logCPM_state_vs_rest"] > 0)).sum()),
        "n_down_basic_filter": int(((df["passes_basic_marker_filter"]) & (df["logFC_logCPM_state_vs_rest"] < 0)).sum()),
    })

all_de = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
test_summary_df = pd.DataFrame(test_summaries)
write_tsv_replace(test_summary_df, DE_ALL_SUMMARY_TSV)
display(test_summary_df)


In [ ]:
# =========================
# Export top 100 up/down and significant tables
# =========================

if all_de.empty:
    raise ValueError("No DE results available.")

# Top 100 up and top 100 down per state.
top_rows = []
for st in valid_states:
    sub = all_de[all_de["malignant_state"] == st].copy()
    if sub.empty:
        continue
    up = (
        sub[sub["logFC_logCPM_state_vs_rest"] > 0]
        .sort_values(["padj_fdr_bh", "pval_mannwhitney", "logFC_logCPM_state_vs_rest"], ascending=[True, True, False])
        .head(TOP_N_PER_DIRECTION)
        .copy()
    )
    up["top_direction"] = "top_up_in_state"
    down = (
        sub[sub["logFC_logCPM_state_vs_rest"] < 0]
        .sort_values(["padj_fdr_bh", "pval_mannwhitney", "logFC_logCPM_state_vs_rest"], ascending=[True, True, True])
        .head(TOP_N_PER_DIRECTION)
        .copy()
    )
    down["top_direction"] = "top_down_in_state"
    top_rows.extend([up, down])

top_df = pd.concat(top_rows, ignore_index=True)
write_tsv_replace(top_df, DE_TOP100_TSV)
display(top_df.head(20))

sig_df = all_de[all_de["passes_basic_marker_filter"]].copy()
sig_df = sig_df.sort_values(["malignant_state", "padj_fdr_bh", "pval_mannwhitney", "logFC_logCPM_state_vs_rest"], ascending=[True, True, True, False])
write_tsv_replace(sig_df, DE_SIGNIFICANT_TSV)
print("significant rows:", sig_df.shape)


In [ ]:
# =========================
# Marker refinement summary
# =========================

summary_rows = []
for st in valid_states:
    sub = all_de[all_de["malignant_state"] == st].copy()
    sig = sub[sub["passes_basic_marker_filter"]].copy()
    up = sig[sig["logFC_logCPM_state_vs_rest"] > 0].sort_values(
        ["padj_fdr_bh", "logFC_logCPM_state_vs_rest"], ascending=[True, False]
    )
    down = sig[sig["logFC_logCPM_state_vs_rest"] < 0].sort_values(
        ["padj_fdr_bh", "logFC_logCPM_state_vs_rest"], ascending=[True, True]
    )
    summary_rows.append({
        "malignant_state": st,
        "n_pseudobulk_state": int(sub["n_pseudobulk_state"].iloc[0]) if len(sub) else 0,
        "n_pseudobulk_rest": int(sub["n_pseudobulk_rest"].iloc[0]) if len(sub) else 0,
        "n_up_markers_padj_lt_0_05_abslogfc_ge_0_25": int(up.shape[0]),
        "n_down_markers_padj_lt_0_05_abslogfc_ge_0_25": int(down.shape[0]),
        "top10_up_genes": ",".join(up.head(10)["gene"].astype(str).tolist()),
        "top10_down_genes": ",".join(down.head(10)["gene"].astype(str).tolist()),
        "max_logFC_gene": sub.sort_values("logFC_logCPM_state_vs_rest", ascending=False).head(1)["gene"].iloc[0] if len(sub) else "NA",
        "max_logFC": float(sub["logFC_logCPM_state_vs_rest"].max()) if len(sub) else np.nan,
        "min_logFC_gene": sub.sort_values("logFC_logCPM_state_vs_rest", ascending=True).head(1)["gene"].iloc[0] if len(sub) else "NA",
        "min_logFC": float(sub["logFC_logCPM_state_vs_rest"].min()) if len(sub) else np.nan,
    })

refinement_summary = pd.DataFrame(summary_rows)
write_tsv_replace(refinement_summary, REFINEMENT_SUMMARY_TSV)
display(refinement_summary)


In [ ]:
# =========================
# Figures: pseudobulk marker heatmap
# =========================

figure_rows = []

# Pick top up markers per state for heatmap.
heatmap_genes = []
for st in valid_states:
    sub = top_df[(top_df["malignant_state"] == st) & (top_df["top_direction"] == "top_up_in_state")].copy()
    sub = sub.sort_values(["padj_fdr_bh", "logFC_logCPM_state_vs_rest"], ascending=[True, False]).head(10)
    heatmap_genes.extend(sub["gene"].tolist())

# De-duplicate while preserving order.
seen = set()
heatmap_genes = [g for g in heatmap_genes if not (g in seen or seen.add(g))]
heatmap_genes = [g for g in heatmap_genes if g in pb_adata.var_names]
print("n heatmap genes:", len(heatmap_genes))

if len(heatmap_genes) > 0:
    # Mean logCPM per state.
    mean_rows = []
    for st in valid_states:
        mask = (pb_adata.obs[state_norm_col].astype(str).values == st) & (pb_adata.obs["n_cells"].values >= MIN_CELLS_PER_PSEUDOBULK)
        if mask.sum() == 0:
            continue
        vals = np.asarray(pb_adata[mask, heatmap_genes].X).mean(axis=0)
        mean_rows.append(pd.Series(vals, index=heatmap_genes, name=st))
    mean_df = pd.DataFrame(mean_rows)

    # Z-score by gene across states.
    mat = mean_df.values.astype(float)
    mat_z = (mat - np.nanmean(mat, axis=0, keepdims=True)) / (np.nanstd(mat, axis=0, keepdims=True) + 1e-8)

    fig_w = max(10, len(heatmap_genes) * 0.22)
    fig_h = max(4, len(mean_df) * 0.55)
    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(mat_z, aspect="auto")
    plt.colorbar(label="z-scored mean logCPM")
    plt.yticks(np.arange(len(mean_df.index)), mean_df.index)
    plt.xticks(np.arange(len(heatmap_genes)), heatmap_genes, rotation=90, fontsize=7)
    plt.title("Top pseudobulk state markers: mean logCPM z-score")
    plt.tight_layout()
    fig_path = FIGURES_DIR / "malignant_state_pseudobulk_top_marker_heatmap.png"
    if fig_path.exists():
        fig_path.unlink()
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    figure_rows.append({
        "figure": "malignant_state_pseudobulk_top_marker_heatmap",
        "path": str(fig_path),
        "description": "Heatmap of top pseudobulk up-marker genes per malignant state, using z-scored mean logCPM across states.",
    })

# Also save mean expression matrix for traceability.
if len(heatmap_genes) > 0:
    mean_expr_tsv = METADATA_DIR / "malignant_state_pseudobulk_top_marker_mean_logCPM_matrix_HVG8000_r0_6.tsv"
    mean_df.reset_index(names="malignant_state").to_csv(mean_expr_tsv, sep="\t", index=False)
    print(f"[SAVED] {mean_expr_tsv}")

fig_manifest = pd.DataFrame(figure_rows)
write_tsv_replace(fig_manifest, FIGURE_MANIFEST_TSV)
display(fig_manifest)


In [ ]:
# =========================
# Output manifest
# =========================

outputs = [
    ("READINESS_TSV", READINESS_TSV),
    ("PSEUDOBULK_QC_TSV", PSEUDOBULK_QC_TSV),
    ("STATE_SUMMARY_TSV", STATE_SUMMARY_TSV),
    ("PSEUDOBULK_H5AD", PSEUDOBULK_H5AD),
    ("DE_TOP100_TSV", DE_TOP100_TSV),
    ("DE_SIGNIFICANT_TSV", DE_SIGNIFICANT_TSV),
    ("DE_ALL_SUMMARY_TSV", DE_ALL_SUMMARY_TSV),
    ("REFINEMENT_SUMMARY_TSV", REFINEMENT_SUMMARY_TSV),
    ("FIGURE_MANIFEST_TSV", FIGURE_MANIFEST_TSV),
]

manifest = pd.DataFrame([{
    "label": label,
    "path": str(path),
    "exists": Path(path).exists(),
    "size_mb": round(Path(path).stat().st_size / 1024 / 1024, 3) if Path(path).exists() else np.nan,
} for label, path in outputs])

write_tsv_replace(manifest, OUTPUT_MANIFEST_TSV)
display(manifest)

print("Done: malignant-state pseudobulk DE / marker refinement.")
